### Training Reasoning Models with Reinforcement Learning
** rlvr-grpo (without batching)

In [1]:
from pathlib import Path
import sys
import torch

ROOT_DIR = Path.cwd().parent  # Get parent of current directory
if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))


from evaluating_reasoning_models.model_and_tokenizer import load_model_and_tokenizer

model, tokenizer = load_model_and_tokenizer(
    which_model="base",
    use_compile=False
)


Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/1.50G [00:00<?, ?B/s]

config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]


Loading base Qwen weights


In [2]:
from improving_reasoning_with_inference_time_scaling.improving_reasoning_with_inference_time_scaling import (
    generate_text_stream_concat_flex,
    generate_text_top_p_stream_cache,
)
from evaluating_reasoning_models.evaluating_reasoning_models import render_prompt

device = "cuda" if torch.cuda.is_available() else "cpu"
raw_prompt = (
    "Half the value of $3x-9$ is $x+37$. "
    "What is the value of $x$?"
)




prompt = render_prompt(raw_prompt)
print(prompt)

torch.manual_seed(0)
response_1 = generate_text_stream_concat_flex(
    model, tokenizer, prompt, device,
    max_new_tokens=1048, verbose=True,
    generate_func=generate_text_top_p_stream_cache,
    temperature=0.2,
    top_p=0.9,
)


You are a helpful math assistant.

Solve the problem step by step.
The last line of your response should contain only the final answer inside \boxed{}.

Question:
Half the value of $3x-9$ is $x+37$. What is the value of $x$?

Answer:

Let's solve the equation step by step:

We are told that **half the value of $3x - 9$ is $x + 37$**.

This means:

$$
\frac{1}{2}(3x - 9) = x + 37
$$

Multiply both sides by 2 to eliminate the fraction:

$$
3x - 9 = 2x + 74
$$

Subtract $2x$ from both sides:

$$
x - 9 = 74
$$

Add 9 to both sides:

$$
x = 83
$$

### Final Answer:
$$
\boxed{83}


#### Loading a Math training subset

In [3]:
import json
import requests
from pathlib import Path

def load_math_train(local_path="math_train.json", save_copy=True):
    local_path = Path(local_path)

    url = (
        "https://raw.githubusercontent.com/rasbt/"
        "math_full_minus_math500/refs/heads/main/"
        "math_full_minus_math500.json"
    )
    backup_url = (
        "https://f001.backblazeb2.com/file/reasoning-from-scratch/"
        "MATH/math_full_minus_math500.json"
    )

    if local_path.exists():
        with local_path.open("r", encoding="utf-8") as f:
            data = json.load(f)
    else:
        try:
            r = requests.get(url, timeout=30)
            r.raise_for_status()
        except requests.RequestException:
            print("Using backup URL.")
            r = requests.get(backup_url, timeout=30)
            r.raise_for_status()

        data = r.json()

        if save_copy:
            with local_path.open("w", encoding="utf-8") as f:
                json.dump(data, f, indent=2)

    return data


In [4]:

math_train = load_math_train()

print("Dataset size:", len(math_train))

Dataset size: 12000


In [5]:
from pprint import pprint

pprint(math_train[4])

{'answer': '6',
 'level': 'Level 3',
 'problem': 'Sam is hired for a 20-day period. On days that he works, he earns '
            '$\\$$60. For each day that he does not work, $\\$$30 is '
            'subtracted from his earnings. At the end of the 20-day period, he '
            'received $\\$$660. How many days did he not work?',
 'solution': 'Call $x$ the number of days Sam works and $y$ the number of days '
             'he does not. We can set up the following system of equations to '
             'represent the given information: \\begin{align*}\n'
             'x+y &= 20 \\\\\n'
             '60x - 30y &= 660 \\\\\n'
             '\\end{align*} The first equation represents the total number of '
             'days Sam works, and the second equation represents his total '
             'profit. Solving for $x$ in the first equation yields $x = 20 - '
             'y$. Substituting into the second equation gives $60(20-y) - 30y '
             '= 660$. Canceling a factor of $10$ an

#### Sampling rollouts

In [6]:
from base_model.qwen import KVCache
from improving_reasoning_with_inference_time_scaling.improving_reasoning_with_inference_time_scaling import top_p_filter
from evaluating_reasoning_models.evaluating_reasoning_models import has_complete_boxed_answer



@torch.no_grad()
def sample_response(
    model,
    tokenizer,
    prompt,
    device,
    max_new_tokens=512,
    temperature=0.8,
    top_p=0.9,
):
    input_ids = torch.tensor(
        tokenizer.encode(prompt),
        device=device
        )

    cache = KVCache(n_layers=model.cfg["n_layers"])
    model.reset_kv_cache()
    logits = model(input_ids.unsqueeze(0), cache=cache)[:, -1]

    generated = []
    for _ in range(max_new_tokens):
        if temperature and temperature != 1.0:
            logits = logits / temperature

        probas = torch.softmax(logits, dim=-1)
        probas = top_p_filter(probas, top_p)
        next_token = torch.multinomial(
            probas.cpu(), num_samples=1
        ).to(device)

        token_id = next_token.item()
        if (
            tokenizer.eos_token_id is not None
            and token_id == tokenizer.eos_token_id
        ):
            break

        decoded = tokenizer.decode(generated)
        if has_complete_boxed_answer(decoded):
            break

        generated.append(token_id)
        
        

        

        logits = model(next_token, cache=cache)[:, -1]

    full_token_ids = torch.cat(
        [input_ids,
         torch.tensor(generated, device=device, dtype=input_ids.dtype),]
    )
    return full_token_ids, input_ids.numel(), tokenizer.decode(generated)

In [7]:
torch.manual_seed(0)

raw_prompt = (
    "Half the value of $3x-9$ is $x+37$. "
    "What is the value of $x$?"
)
prompt = render_prompt(raw_prompt)

token_ids, prompt_len, answer_text = sample_response(
            model=model,
            tokenizer=tokenizer,
            prompt=prompt,
            device=device,
            max_new_tokens=512,
            temperature=0.9,
            top_p=0.9,
        )

print(answer_text)

Given that half the value of $3x - 9$ is $x + 37$, we can set up the equation:

$$
\frac{1}{2}(3x - 9) = x + 37
$$

Multiply both sides by 2 to eliminate the fraction:

$$
3x - 9 = 2x + 74
$$

Subtract $2x$ from both sides:

$$
x - 9 = 74
$$

Add 9 to both sides:

$$
x = 83
$$

The value of $x$ is $\boxed{83}


In [8]:
### example rollouts
rollouts = [
    r"\boxed{83}",
    r"The correct answer is \boxed{83}",
    r"The final answer is 83",
    r"We get \boxed{38}",
]

#### Calculating rewards

In [9]:
from evaluating_reasoning_models.evaluating_reasoning_models import (extract_final_candidate,
grade_answer)

def reward_rlvr(answer_text,  ground_truth):
    extracted = extract_final_candidate(answer_text, fallback=None)

    if not extracted:
        return 0.0
    
    correct = grade_answer(extracted, ground_truth)
    return float(correct)


In [10]:

rollout_rewards = []

for answer in rollouts:
    reward = reward_rlvr(answer_text=answer, ground_truth="83")
    print(f"Answer: {answer!r}")
    print(f"Reward: {reward}\n")
    rollout_rewards.append(reward)

Answer: '\\boxed{83}'
Reward: 1.0

Answer: 'The correct answer is \\boxed{83}'
Reward: 1.0

Answer: 'The final answer is 83'
Reward: 0.0

Answer: 'We get \\boxed{38}'
Reward: 0.0



#### Learning signals from rollouts via advantages

In [ ]:
rewards = torch.tensor(rollout_rewards, device=device)
print(rewards)

tensor([1., 1., 0., 0.], device='cuda:0')


In [12]:
advantages = (rewards - rewards.mean()) / (rewards.std() + 1e-4)

print(advantages)

tensor([ 0.8659,  0.8659, -0.8659, -0.8659], device='cuda:0')


#### Scoring rollouts with sequence log-probabilites

** using torch.gather function

In [ ]:

def sequence_logprob(model, token_ids, prompt_len):
    logits = model(token_ids.unsqueeze(0)).squeeze(0).float()
    logprobs = torch.log_softmax(logits, dim=-1)
    selected = logprobs[:-1].gather(
        1, token_ids[1:].unsqueeze(-1)
    ).squeeze(-1)
    return torch.sum(selected[prompt_len - 1:])

print(sequence_logprob(model, token_ids, prompt_len))

tensor(-7.0412, device='cuda:0', grad_fn=<SumBackward0>)


In [ ]:
rollouts = [
    r"\boxed{83}",
    r"The correct answer is \boxed{83}",
    r"The final answer is 83",
    r"We get \boxed{38}",
]

rollout_logps = []

for text in rollouts:
    token_ids = tokenizer.encode(prompt + " " + text)
    logprob = sequence_logprob(
        model=model,
        token_ids=torch.tensor(token_ids, device=device),
        prompt_len=prompt_len,
    )

    print(f"Answer:  {text}")
    print(f"Logprob: {logprob.item():.4f}\n")

    rollout_logps.append(logprob)

Answer:  \boxed{83}
Logprob: -59.7055

Answer:  The correct answer is \boxed{83}
Logprob: -82.3536

Answer:  The final answer is 83
Logprob: -60.0746

Answer:  We get \boxed{38}
Logprob: -58.6562



#### from advatages to policy-updates via a grpo loss

#### Putting everything together in a GRPO step

In [15]:
def compute_grpo_loss(
    model,
    tokenizer,
    example,
    device,
    num_rollouts=4,
    max_new_tokens=300,
    temperature=0.8,
    top_p=0.9,
):
    assert num_rollouts >= 2
    roll_logps, roll_rewards, samples = [], [], []
    prompt = render_prompt(example["problem"])

    was_training = model.training
    model.eval()

    for _ in range(num_rollouts):
        # Stage 1: generate rollouts
        token_ids, prompt_len, text = sample_response(
            model=model,
            tokenizer=tokenizer,
            prompt=prompt,
            device=device,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            top_p=top_p,
        )
        # Stage 2: compute rewards
        reward = reward_rlvr(text, example["answer"])
        
        # Stage 4: compute logprobs
        logp = sequence_logprob(model, token_ids, prompt_len)

        roll_logps.append(logp)
        roll_rewards.append(reward)
        samples.append(
            {
                "text": text,
                "reward": reward,
                "gen_len": token_ids.numel() - prompt_len,
            }
        )

    if was_training:
        model.train()

    # Stage 2: collect all rewards
    rewards = torch.tensor(roll_rewards, device=device)

    # Stage 3: compute advantages
    advantages = (rewards - rewards.mean()) / (rewards.std() + 1e-4)

    # Stage 4: collect all logprobs
    logps = torch.stack(roll_logps)

    # Stage 5: compute policy gradient loss
    pg_loss = -(advantages.detach() * logps).mean()
    loss = pg_loss  # In the next chapter we add a KL term here

    return {
        "loss": loss.item(),
        "pg_loss": pg_loss.item(),
        "rewards": roll_rewards,
        "advantages": advantages.detach().cpu().tolist(),
        "samples": samples,
        "loss_tensor": loss,
    }

In [16]:

torch.manual_seed(123)

## compute grpo loss
stats = compute_grpo_loss(
    model=model,
    tokenizer=tokenizer,
    example=math_train[4],
    device=device,
    num_rollouts=4,
    max_new_tokens=300,
    temperature=0.8,
    top_p=0.9
)

pprint(stats)

{'advantages': [0.0, 0.0, 0.0, 0.0],
 'loss': -0.0,
 'loss_tensor': tensor(-0., device='cuda:0', grad_fn=<NegBackward0>),
 'pg_loss': -0.0,
 'rewards': [0.0, 0.0, 0.0, 0.0],
 'samples': [{'gen_len': 247,
              'reward': 0.0,
              'text': 'Let $ x $ be the number of days Sam worked. Then, since '
                      'there are 20 days in total, the number of days he did '
                      'not work is $ 20 - x $.\n'
                      '\n'
                      'He earned $60 per day he worked, so his total earnings '
                      'from working days are $ 60x $. \n'
                      '\n'
                      'He also earned $30 per day he did not work, so his '
                      'total earnings from non-working days are $ 30(20 - x) '
                      '$.\n'
                      '\n'
                      'We are told that the total amount he received is $660, '
                      'so:\n'
                      '\n'
                 

#### Implementating the GRPO training loop

In [17]:
import time

def train_rlvr_grpo(
    model,
    tokenizer,
    math_data,
    device,
    steps=None,
    num_rollouts=4,
    max_new_tokens=300,
    temperature=0.8,
    top_p=0.9,
    lr=1e-5,
    checkpoint_every=50,
    checkpoint_dir=".",
    csv_log_path=None,

):
    if steps is None:
        steps = len(math_data)

    # Stage 1: initialize optimizer
    # (the model was already initialized outside the function)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
    model.train()
    current_step = 0
    if csv_log_path is None:
        timestamp = time.strftime("%Y%m%d_%H%M%S")
        csv_log_path = f"train_rlvr_grpo_metrics_{timestamp}.csv"
    csv_log_path = Path(csv_log_path)

    try:
        # Stage 2: Iterate over training steps
        for step in range(steps):

            # Stage 3: Reset loss gradient
            # (it's best practice to do this at the beginning of each step)
            optimizer.zero_grad()

            current_step = step + 1
            example = math_data[step % len(math_data)]

            # Stage 4: calculate GRPO loss
            stats = compute_grpo_loss(
                model=model,
                tokenizer=tokenizer,
                example=example,
                device=device,
                num_rollouts=num_rollouts,
                max_new_tokens=max_new_tokens,
                temperature=temperature,
                top_p=top_p,
            )

            # Stage 5: Backward pass to calculate loss gradients
            stats["loss_tensor"].backward()

            # Clip large gradients to improve training stability
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

            # Stage 6: Update model weights using loss gradients
            optimizer.step()

            # Stage 7: Collect rewards, response lengths, and losses
            reward_avg = torch.tensor(stats["rewards"]).mean().item()
            step_tokens = sum(
                sample["gen_len"] for sample in stats["samples"]
            )
            avg_response_len = (
                step_tokens / len(stats["samples"]) 
                if stats["samples"] else 0.0
            )
            append_csv_metrics(
                csv_log_path, current_step, steps, stats["loss"],
                reward_avg, avg_response_len,
            )

            # Print step metrics
            print(
                f"[Step {current_step}/{steps}] "
                f"loss={stats['loss']:.4f} "
                f"reward_avg={reward_avg:.3f} "
                f"avg_resp_len={avg_response_len:.1f}"
            )

            # Sample outputs (every 10 steps) to check if model
            # generates coherent text
            if current_step % 10 == 0:
                print(f"[Step {current_step}] sample outputs")
                for i, sample in enumerate(stats["samples"][:3]):
                    text = sample["text"].replace("\n", "\\n")
                    print(
                        f"  {i+1}) reward={sample['reward']:.3f} "
                        f"len={sample['gen_len']}: {text}"
                    )
                print()

            # Stage 8: Save model checkpoint
            if checkpoint_every and current_step % checkpoint_every == 0:
                ckpt_path = save_checkpoint(
                    model=model,
                    checkpoint_dir=checkpoint_dir,
                    step=current_step,
                )
                print(f"Saved checkpoint to {ckpt_path}")

    # Save a model checkpoint if we interrupt the training early
    except KeyboardInterrupt:
        ckpt_path = save_checkpoint(
            model=model,
            checkpoint_dir=checkpoint_dir,
            step=max(1, current_step),
            suffix="interrupt",
        )
        print(f"\nKeyboardInterrupt. Saved checkpoint to {ckpt_path}")
        return model

    return model

from safetensors.torch import save_file

def save_checkpoint(model, checkpoint_dir, step, suffix=""):
    checkpoint_dir = Path(checkpoint_dir)
    checkpoint_dir.mkdir(parents=True, exist_ok=True)

    suffix = f"-{suffix}" if suffix else ""

    ckpt_path = (
        checkpoint_dir /
        f"qwen3-0.6B-rlvr-grpo-step{step:05d}{suffix}.safetensors"
    )

    save_file(model.state_dict(), str(ckpt_path))

    return ckpt_path


def append_csv_metrics(
    csv_log_path,
    step_idx,
    total_steps,
    loss,
    reward_avg,
    avg_response_len,
):
    if not csv_log_path.exists():
        csv_log_path.write_text(
            "step,total_steps,loss,reward_avg,avg_response_len\n",
            encoding="utf-8",
        )
    with csv_log_path.open("a", encoding="utf-8") as f:
        f.write(
            f"{step_idx},{total_steps},{loss:.6f},{reward_avg:.6f},"
            f"{avg_response_len:.6f}\n"
        )

In [18]:
## running the loop
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

torch.manual_seed(0)

train_rlvr_grpo(
    model=model,
    tokenizer=tokenizer,
    math_data=math_train,
    device=device,
    steps=50,
    num_rollouts=4,
    max_new_tokens=300,
    temperature=0.8,
    top_p=0.9,
    lr=1e-5,
    checkpoint_every=5,
    checkpoint_dir=".",
    csv_log_path="train_rlvr_grpo_metrics.csv",
)

[Step 1/50] loss=-1.2391 reward_avg=0.250 avg_resp_len=287.8
[Step 2/50] loss=-3.2039 reward_avg=0.250 avg_resp_len=295.0
[Step 3/50] loss=-0.0000 reward_avg=1.000 avg_resp_len=152.2
[Step 4/50] loss=-0.0000 reward_avg=1.000 avg_resp_len=129.0
[Step 5/50] loss=-5.9458 reward_avg=0.750 avg_resp_len=143.5
Saved checkpoint to qwen3-0.6B-rlvr-grpo-step00005.safetensors
[Step 6/50] loss=-0.0000 reward_avg=1.000 avg_resp_len=181.8
[Step 7/50] loss=-0.0000 reward_avg=0.000 avg_resp_len=300.0
[Step 8/50] loss=-0.0000 reward_avg=1.000 avg_resp_len=134.2


OutOfMemoryError: CUDA out of memory. Tried to allocate 322.00 MiB. GPU 0 has a total capacity of 14.56 GiB of which 270.81 MiB is free. Including non-PyTorch memory, this process has 14.29 GiB memory in use. Of the allocated memory 13.89 GiB is allocated by PyTorch, and 272.31 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

#### Loading and evaluating saved model checkpoints

# Evaluating RLVR-GRPO Checkpoints

The model loader now supports both the base Qwen model and RLVR-GRPO checkpoints from Hugging Face.

## Base Model

```python
model, tokenizer = load_model_and_tokenizer(
    which_model="base",
    use_compile=False,
    load_rlvr_checkpoint=False,
)
```

This downloads and loads:

```text
Qwen/Qwen3-0.6B
```

---

## RLVR-GRPO Checkpoint

```python
model, tokenizer = load_model_and_tokenizer(
    which_model="base",
    use_compile=False,
    load_rlvr_checkpoint=True,
)
```

This automatically:

1. Downloads the configured RLVR-GRPO repository from Hugging Face.
2. Downloads the checkpoint, tokenizer, and config files.
3. Loads the checkpoint directly into the custom `Qwen3Model`.

---

## Result

A single loader function can now switch between:

- Base Qwen3-0.6B
- RLVR-GRPO checkpoints

without requiring any manual code changes.

In [1]:
from pathlib import Path
import sys
import torch

ROOT_DIR = Path.cwd().parent  # Get parent of current directory
if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))


from evaluating_reasoning_models.model_and_tokenizer import load_model_and_tokenizer

model, tokenizer = load_model_and_tokenizer(
    which_model="base",
    use_compile=False,
    load_rlvr_checkpoint=True
)


Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]


Loading RLVR checkpoint: /teamspace/studios/this_studio/open-posttraining-system/training_reasoning_models_with_reinforcement_learning/qwen/qwen3-0.6B-rlvr-grpo-step00005.safetensors
Loaded 311 tensors
Missing keys: 0
Unexpected keys: 0


In [20]:
### on base model--> 
from evaluating_reasoning_models.evaluating_reasoning_models import evaluate_math500_stream

math_eval = math_train[450:500]  ## last 50 problems of math_dataset...


## evaluation function
num_correct, num_examples, acc = evaluate_math500_stream(model, tokenizer, device, math_eval, max_new_tokens=2048, verbose=False)




Progress: 50/50 | ETA: 00s        

Accuracy: 64.0% (32/50)
Total time: 7.8 minutes
Average response length: 303.0 tokens
Results saved to: math500-cuda.jsonl


In [2]:
import torch
device = "cuda" if torch.cuda.is_available() else "cpu"

In [7]:
### on rlvr_grpo  checkpoing model--> (5steps ---> 4 rollouts)
from evaluating_reasoning_models.evaluating_reasoning_models import evaluate_math500_stream

math_eval = math_train[450:500]  ## last 50 problems of math_dataset...


## evaluation function
num_correct, num_examples, acc = evaluate_math500_stream(model, tokenizer, device, math_eval, max_new_tokens=2048, verbose=False)


Progress: 50/50 | ETA: 00s        

Accuracy: 48.0% (24/50)
Total time: 5.9 minutes
Average response length: 223.4 tokens
Results saved to: math500-cuda.jsonl


#### rlvr_grpo checkpoint performed worst than base-model on 5 steps..
* now increasing the rollouts = 8, and steps = 50  --> from Tesla 4 ---> l40s gpu..

In [18]:
## running the loop
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

torch.manual_seed(0)

train_rlvr_grpo(
    model=model,
    tokenizer=tokenizer,
    math_data=math_train,
    device=device,
    steps=50,
    num_rollouts=8,
    max_new_tokens=300,
    temperature=0.8,
    top_p=0.9,
    lr=1e-5,
    checkpoint_every=5,
    checkpoint_dir=".",
    csv_log_path="train_rlvr_grpo_metrics.csv",
)

[Step 1/50] loss=-0.0000 reward_avg=0.000 avg_resp_len=297.4
[Step 2/50] loss=-0.0000 reward_avg=0.000 avg_resp_len=289.0
[Step 3/50] loss=-0.0000 reward_avg=1.000 avg_resp_len=178.6
[Step 4/50] loss=-0.0000 reward_avg=1.000 avg_resp_len=146.2
[Step 5/50] loss=-2.0220 reward_avg=0.250 avg_resp_len=187.8
Saved checkpoint to qwen3-0.6B-rlvr-grpo-step00005.safetensors
[Step 6/50] loss=-3.4400 reward_avg=0.375 avg_resp_len=282.9
[Step 7/50] loss=-0.0000 reward_avg=0.000 avg_resp_len=298.5
[Step 8/50] loss=-0.0000 reward_avg=1.000 avg_resp_len=183.2
[Step 9/50] loss=-0.0000 reward_avg=0.000 avg_resp_len=260.4
[Step 10/50] loss=-0.0000 reward_avg=0.000 avg_resp_len=292.8
[Step 10] sample outputs
  1) reward=0.000 len=299: To find out how much money Tim should invest, we use the compound interest formula:\n\n$$\nA = P \left(1 + \frac{r}{n}\right)^{nt}\n$$\n\nWhere:\n- $ A = 60,000 $ (the final amount),\n- $ r = 0.07 $ (the annual interest rate),\n- $ n = 4 $ (since it compounds quarterly),\n-

Qwen3Model(
  (tok_emb): Embedding(151936, 1024)
  (trf_blocks): ModuleList(
    (0-27): 28 x TransformerBlock(
      (att): GroupQueryAttention(
        (w_query): Linear(in_features=1024, out_features=2048, bias=False)
        (w_keys): Linear(in_features=1024, out_features=1024, bias=False)
        (w_values): Linear(in_features=1024, out_features=1024, bias=False)
        (proj_out): Linear(in_features=2048, out_features=1024, bias=False)
        (q_norm): RMSNorm()
        (k_norm): RMSNorm()
      )
      (ff): FeedForward(
        (fc1): Linear(in_features=1024, out_features=3072, bias=False)
        (fc2): Linear(in_features=1024, out_features=3072, bias=False)
        (fc3): Linear(in_features=3072, out_features=1024, bias=False)
      )
      (rms_norm1): RMSNorm()
      (rms_norm2): RMSNorm()
    )
  )
  (final_norm): RMSNorm()
  (out_head): Linear(in_features=1024, out_features=151936, bias=False)
)

In [1]:
##  changed the checkpoint url--> in model_and_tokenizer and downlaod..file.. see  those.. next time will support every checkpoint by-default(few changes in code)..
from pathlib import Path
import sys
import torch

ROOT_DIR = Path.cwd().parent  # Get parent of current directory
if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))


from evaluating_reasoning_models.model_and_tokenizer import load_model_and_tokenizer

model, tokenizer = load_model_and_tokenizer(
    which_model="base",
    use_compile=False,
    load_rlvr_checkpoint=True
)


Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

qwen3-0.6B-rlvr-grpo-step00050.safetenso(…):   0%|          | 0.00/1.50G [00:00<?, ?B/s]


Loading RLVR checkpoint: /teamspace/studios/this_studio/open-posttraining-system/training_reasoning_models_with_reinforcement_learning/qwen/qwen3-0.6B-rlvr-grpo-step00050.safetensors
Loaded 311 tensors
Missing keys: 0
Unexpected keys: 0


In [5]:
## ### on rlvr_grpo checkpoint(8 rollouts  ---> 50 steps)--> 
from evaluating_reasoning_models.evaluating_reasoning_models import evaluate_math500_stream

math_eval = math_train[450:500]  ## last 50 problems of math_dataset...


## evaluation function
num_correct, num_examples, acc = evaluate_math500_stream(model, tokenizer, device, math_eval, max_new_tokens=2048, verbose=False)




Progress: 50/50 | ETA: 00s        

Accuracy: 60.0% (30/50)
Total time: 3.9 minutes
Average response length: 219.2 tokens
Results saved to: math500-cuda.jsonl


# Notes / Initial Observations

## Evaluation Results

### Base Model

- Correct: **32 / 50**
- Accuracy: **64.0%**
- Total evaluation time: **7.8 minutes**
- Average response length: **303.0 tokens**

### RLVR-GRPO (5-Step Checkpoint)

Training configuration:

- GRPO steps: **5**
- Rollouts: **4**
- Max generation length: **300 tokens**
- Sparse binary reward (correct answer = 1, incorrect answer = 0)

Evaluation results:

- Correct: **24 / 50**
- Accuracy: **48.0%**
- Total evaluation time: **5.9 minutes**
- Average response length: **223.4 tokens**

### RLVR-GRPO (50-Step Checkpoint)

Training configuration:

- GRPO steps: **50**
- Rollouts: **8**
- Max generation length: **300 tokens**
- Sparse binary reward (correct answer = 1, incorrect answer = 0)

Evaluation results:

- Correct: **30 / 50**
- Accuracy: **60.0%**
- Total evaluation time: **3.9 minutes**
- Average response length: **219.2 tokens**

---

## Initial Observations

- The 5-step RLVR-GRPO checkpoint underperformed the base model, dropping from **64% → 48%** accuracy.
- The 50-step RLVR-GRPO checkpoint recovered much of the lost performance, improving from **48% → 60%** accuracy.
- The final RLVR-GRPO checkpoint remained slightly below the base model (**60% vs 64%**), although the difference corresponds to only **2 questions** on a 50-problem evaluation set.
- Response lengths decreased substantially after RL training (**303 → ~220 tokens**), suggesting the policy learned shorter solution trajectories.
- Evaluation latency also decreased as response lengths became shorter.
- Training used a sparse reward signal based only on final answer correctness, providing limited learning signal compared to denser reward formulations.
- These results suggest that the GRPO implementation is functioning correctly and that learning occurred during training. However, additional training steps, larger evaluation sets, and improved reward design are likely needed to determine whether RLVR-GRPO can consistently outperform the base model.